# Titanic - Feature Engineering

Neste notebook vamos preparar os dados para modelagem:
- Tratar valores faltantes
- Criar novas features
- Fazer encoding de variáveis categóricas
- Normalizar/padronizar quando necessário
- Salvar datasets processados

## 1. Setup e Carregamento dos Dados

In [ ]:
# Importar bibliotecas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("✅ Bibliotecas importadas com sucesso!")

In [ ]:
# Carregar datasets
train = pd.read_csv('dados/train.csv')
test = pd.read_csv('dados/test.csv')

# Guardar PassengerId para submissão
test_ids = test['PassengerId'].copy()

print(f"📊 Train: {train.shape}")
print(f"📊 Test: {test.shape}")
print(f"\n✅ Dados carregados!")

## 2. Análise de Valores Faltantes

In [ ]:
# Função para analisar valores faltantes
def check_missing(df, name):
    print(f"\n🔍 Valores Faltantes - {name}")
    print("="*60)
    missing = df.isnull().sum()
    missing = missing[missing > 0].sort_values(ascending=False)
    
    if len(missing) == 0:
        print("✅ Nenhum valor faltante!")
    else:
        missing_df = pd.DataFrame({
            'Coluna': missing.index,
            'Faltantes': missing.values,
            'Percentual (%)': (missing.values / len(df) * 100).round(2)
        })
        print(missing_df.to_string(index=False))

check_missing(train, 'TREINO')
check_missing(test, 'TESTE')

## 3. Feature Engineering

### 3.1. Extrair Título do Nome

In [ ]:
# Extrair título do nome
def extract_title(df):
    df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
    return df

train = extract_title(train)
test = extract_title(test)

print("👑 Títulos encontrados no TREINO:")
print(train['Title'].value_counts())
print("\n👑 Títulos encontrados no TESTE:")
print(test['Title'].value_counts())

In [ ]:
# Simplificar títulos raros
def simplify_titles(df):
    # Mapear títulos raros para categorias mais comuns
    title_mapping = {
        'Mr': 'Mr',
        'Miss': 'Miss',
        'Mrs': 'Mrs',
        'Master': 'Master',
        'Dr': 'Rare',
        'Rev': 'Rare',
        'Col': 'Rare',
        'Major': 'Rare',
        'Mlle': 'Miss',  # Mademoiselle = Miss em francês
        'Mme': 'Mrs',    # Madame = Mrs em francês
        'Don': 'Rare',
        'Dona': 'Rare',
        'Lady': 'Rare',
        'Countess': 'Rare',
        'Jonkheer': 'Rare',
        'Sir': 'Rare',
        'Capt': 'Rare',
        'Ms': 'Miss'
    }
    
    df['Title'] = df['Title'].map(title_mapping)
    # Se algum título não foi mapeado, colocar como 'Rare'
    df['Title'].fillna('Rare', inplace=True)
    return df

train = simplify_titles(train)
test = simplify_titles(test)

print("✅ Títulos simplificados!")
print("\nTREINO:")
print(train['Title'].value_counts())
print("\nTESTE:")
print(test['Title'].value_counts())

### 3.2. Criar Features de Família

In [ ]:
# Criar Family_Size e IsAlone
def create_family_features(df):
    # Tamanho da família (incluindo o passageiro)
    df['Family_Size'] = df['SibSp'] + df['Parch'] + 1
    
    # Está sozinho?
    df['IsAlone'] = (df['Family_Size'] == 1).astype(int)
    
    return df

train = create_family_features(train)
test = create_family_features(test)

print("✅ Features de família criadas!")
print(f"\nFamily_Size - TREINO:")
print(train['Family_Size'].value_counts().sort_index())
print(f"\nIsAlone - TREINO:")
print(train['IsAlone'].value_counts())

### 3.3. Tratar Idade (Age)

In [ ]:
# Preencher Age com a mediana por Pclass e Title
def fill_age(df):
    # Calcular mediana de idade por Pclass e Title
    age_by_group = df.groupby(['Pclass', 'Title'])['Age'].transform('median')
    
    # Preencher valores faltantes
    df['Age'] = df['Age'].fillna(age_by_group)
    
    # Se ainda houver NaN (caso raro), preencher com mediana geral
    df['Age'].fillna(df['Age'].median(), inplace=True)
    
    return df

train = fill_age(train)
test = fill_age(test)

print("✅ Idade preenchida!")
print(f"Valores faltantes em Age (TREINO): {train['Age'].isnull().sum()}")
print(f"Valores faltantes em Age (TESTE): {test['Age'].isnull().sum()}")

### 3.4. Criar Grupos de Idade (Age_Group)

In [ ]:
# Criar grupos de idade
def create_age_groups(df):
    df['Age_Group'] = pd.cut(df['Age'], 
                             bins=[0, 12, 18, 60, 100], 
                             labels=['Child', 'Teen', 'Adult', 'Senior'])
    return df

train = create_age_groups(train)
test = create_age_groups(test)

print("✅ Grupos de idade criados!")
print("\nTREINO:")
print(train['Age_Group'].value_counts().sort_index())

### 3.5. Tratar Embarked

In [ ]:
# Preencher Embarked com a moda (valor mais frequente)
def fill_embarked(df):
    df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)
    return df

train = fill_embarked(train)
test = fill_embarked(test)

print("✅ Embarked preenchido!")
print(f"Valores faltantes em Embarked (TREINO): {train['Embarked'].isnull().sum()}")
print(f"Valores faltantes em Embarked (TESTE): {test['Embarked'].isnull().sum()}")

### 3.6. Tratar Fare

In [ ]:
# Preencher Fare com a mediana da classe
def fill_fare(df):
    df['Fare'].fillna(df.groupby('Pclass')['Fare'].transform('median'), inplace=True)
    return df

train = fill_fare(train)
test = fill_fare(test)

print("✅ Fare preenchido!")
print(f"Valores faltantes em Fare (TREINO): {train['Fare'].isnull().sum()}")
print(f"Valores faltantes em Fare (TESTE): {test['Fare'].isnull().sum()}")

### 3.7. Extrair Deck da Cabine

In [ ]:
# Extrair deck (primeira letra) da cabine
def extract_deck(df):
    df['Deck'] = df['Cabin'].str[0]
    # Preencher valores faltantes com 'U' (Unknown)
    df['Deck'].fillna('U', inplace=True)
    return df

train = extract_deck(train)
test = extract_deck(test)

print("✅ Deck extraído!")
print("\nDeck - TREINO:")
print(train['Deck'].value_counts().sort_index())

### 3.8. Criar Grupos de Tarifa (Fare_Group)

In [ ]:
# Criar grupos de tarifa
def create_fare_groups(df):
    df['Fare_Group'] = pd.qcut(df['Fare'], q=4, labels=['Low', 'Medium', 'High', 'Very_High'], duplicates='drop')
    return df

train = create_fare_groups(train)
test = create_fare_groups(test)

print("✅ Grupos de tarifa criados!")
print("\nFare_Group - TREINO:")
print(train['Fare_Group'].value_counts().sort_index())

## 4. Verificar Valores Faltantes Após Feature Engineering

In [ ]:
check_missing(train, 'TREINO - Após FE')
check_missing(test, 'TESTE - Após FE')

# CORREÇÃO FINAL: Garantir que não há NaN
print("\n🔧 Verificando e corrigindo valores NaN finais...")

# Preencher qualquer NaN restante
for col in train.columns:
    if train[col].isnull().sum() > 0:
        if train[col].dtype in ['float64', 'int64']:
            train[col].fillna(train[col].median(), inplace=True)
        else:
            train[col].fillna(train[col].mode()[0], inplace=True)
        print(f"   Corrigido: {col}")

for col in test.columns:
    if test[col].isnull().sum() > 0:
        if test[col].dtype in ['float64', 'int64']:
            test[col].fillna(test[col].median(), inplace=True)
        else:
            test[col].fillna(test[col].mode()[0], inplace=True)
        print(f"   Corrigido: {col}")

print("\n✅ Todos os valores NaN foram tratados!")

## 5. Encoding de Variáveis Categóricas

In [ ]:
# Encoding de variáveis categóricas
def encode_features(df):
    # Sex: male=0, female=1
    df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
    
    # Embarked: S=0, C=1, Q=2
    df['Embarked'] = df['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})
    
    # Title: Mr=0, Miss=1, Mrs=2, Master=3, Rare=4
    df['Title'] = df['Title'].map({'Mr': 0, 'Miss': 1, 'Mrs': 2, 'Master': 3, 'Rare': 4})
    
    # Age_Group: Child=0, Teen=1, Adult=2, Senior=3
    df['Age_Group'] = df['Age_Group'].map({'Child': 0, 'Teen': 1, 'Adult': 2, 'Senior': 3})
    
    # Fare_Group: Low=0, Medium=1, High=2, Very_High=3
    df['Fare_Group'] = df['Fare_Group'].map({'Low': 0, 'Medium': 1, 'High': 2, 'Very_High': 3})
    
    # Deck: A=0, B=1, ..., U=8
    deck_mapping = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6, 'T': 7, 'U': 8}
    df['Deck'] = df['Deck'].map(deck_mapping)
    
    return df

train = encode_features(train)
test = encode_features(test)

print("✅ Encoding realizado!")
print("\nPrimeiras linhas do TREINO após encoding:")
print(train.head())

## 6. Selecionar Features para Modelagem

In [ ]:
# Features que vamos usar para treinar os modelos
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 
            'Title', 'Family_Size', 'IsAlone', 'Age_Group', 'Deck', 'Fare_Group']

# Separar X (features) e y (target)
X_train = train[features]
y_train = train['Survived']

X_test = test[features]

# IMPORTANTE: Verificar se ainda há NaN
print(f"\n🔍 Verificação final de NaN:")
print(f"X_train NaN: {X_train.isnull().sum().sum()}")
print(f"X_test NaN: {X_test.isnull().sum().sum()}")

if X_train.isnull().sum().sum() > 0:
    print("\n⚠️ Ainda há NaN em X_train! Corrigindo...")
    X_train = X_train.fillna(X_train.median())

if X_test.isnull().sum().sum() > 0:
    print("⚠️ Ainda há NaN em X_test! Corrigindo...")
    X_test = X_test.fillna(X_test.median())

print("\n✅ Features selecionadas!")
print(f"\nShape de X_train: {X_train.shape}")
print(f"Shape de y_train: {y_train.shape}")
print(f"Shape de X_test: {X_test.shape}")

print("\nFeatures utilizadas:")
for i, feature in enumerate(features, 1):
    print(f"{i}. {feature}")

## 7. Normalizar Features Numéricas (Opcional)

In [ ]:
from sklearn.preprocessing import StandardScaler

# Criar cópias para versão normalizada (opcional para alguns modelos)
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

# Features numéricas para normalizar
numeric_features = ['Age', 'Fare']

# Aplicar StandardScaler
scaler = StandardScaler()
X_train_scaled[numeric_features] = scaler.fit_transform(X_train[numeric_features])
X_test_scaled[numeric_features] = scaler.transform(X_test[numeric_features])

# Verificação final de NaN após normalização
print(f"\n🔍 Verificação após normalização:")
print(f"X_train_scaled NaN: {X_train_scaled.isnull().sum().sum()}")
print(f"X_test_scaled NaN: {X_test_scaled.isnull().sum().sum()}")

print("\n✅ Normalização aplicada (Age e Fare)!")
print("\nMédia e Desvio Padrão - ANTES da normalização:")
print(X_train[numeric_features].describe())
print("\nMédia e Desvio Padrão - APÓS normalização:")
print(X_train_scaled[numeric_features].describe())

## 8. Salvar Datasets Processados

In [ ]:
# Salvar datasets processados

# Dataset completo (com todas as colunas originais + features criadas)
train.to_csv('train_full_processed.csv', index=False)
test.to_csv('test_full_processed.csv', index=False)

# Dataset para modelagem (apenas features selecionadas)
X_train.to_csv('X_train.csv', index=False)
y_train.to_csv('y_train.csv', index=False)
X_test.to_csv('X_test.csv', index=False)

# Versões normalizadas
X_train_scaled.to_csv('X_train_scaled.csv', index=False)
X_test_scaled.to_csv('X_test_scaled.csv', index=False)

# PassengerId para submissão
test_ids.to_csv('test_ids.csv', index=False)

print("✅ Datasets salvos com sucesso!")
print("\nArquivos criados:")
print("1. train_full_processed.csv - Dataset de treino completo")
print("2. test_full_processed.csv - Dataset de teste completo")
print("3. X_train.csv - Features de treino")
print("4. y_train.csv - Target (Survived)")
print("5. X_test.csv - Features de teste")
print("6. X_train_scaled.csv - Features de treino normalizadas")
print("7. X_test_scaled.csv - Features de teste normalizadas")
print("8. test_ids.csv - PassengerId para submissão")